## Library

In [1]:
import os
import numpy as np
import torch
import librosa
from tqdm import tqdm
from transformers import Wav2Vec2FeatureExtractor, AutoModel

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and Config

In [2]:

AUDIO_DIR = "../Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"     ###Pass the audio file path inside the quotes
SAVE_DIR = "test"     ###Pass the Embeddings storing path inside the quotes
MODEL_NAME = "microsoft/wavlm-base"   ##keep it as it is
SAMPLE_RATE = 16000  ### if you want you can check your audio file sample rates

os.makedirs(SAVE_DIR, exist_ok=True)

## Load Model


In [4]:
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
### we are loading the model from huggin face so dont change anything here.

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████| 248/248 [00:00<00:00, 7226.69it/s]


WavLMModel(
  (feature_extractor): WavLMFeatureEncoder(
    (conv_layers): ModuleList(
      (0): WavLMGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): WavLMFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): WavLMEncoder(
    (pos_conv_embed): WavLMPositionalConvEmbedding(
      (conv): Parametrized

## Process and store the file


In [5]:
counter = 0

X = []
file_names = []
for file in tqdm(os.listdir(AUDIO_DIR)[:10]):
    if(counter>10):
        break
    counter+=1
    if not file.endswith((".wav", ".mp3")):
        continue

    file_path = os.path.join(AUDIO_DIR, file)

    try:
        # Load audio
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        # Process input
        inputs = processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")

        # Extract embeddings
        with torch.no_grad():
            outputs = model(**inputs)

        # Mean pooling
        embedding = outputs.last_hidden_state.mean(dim=1)

        # Convert to numpy
        embedding_np = embedding.squeeze().numpy()

        # Normalize
        embedding_np = embedding_np / np.linalg.norm(embedding_np)

        # Store
        X.append(embedding_np)
        file_names.append(file)

    except Exception as e:
        print(f"Error processing {file}: {e}")

  0%|                                                                                                                      | 0/10 [00:00<?, ?it/s]/home/anshuman139/venv/lib/python3.12/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:26<00:00,  2.66s/it]


## Save in .npy file

In [11]:
X = np.array(X)
np.save(os.path.join(SAVE_DIR, "X.npy"), X)

print("Saved embeddings:", X.shape)

Saved embeddings: (2, 768)


In [14]:
Y = np.load('test/X.npy')